In [1]:
from typing import List, Optional, Union

import gradio as gr
import torch
import transformers
from datasets import Dataset, features, load_dataset
from llavajp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llavajp.conversation import conv_templates
from llavajp.model.llava_gpt2 import LlavaGpt2ForCausalLM
from llavajp.model.llava_llama import LlavaLlamaForCausalLM
from llavajp.train.dataset import tokenizer_image_token
from peft import LoraConfig
from tqdm import tqdm
from transformers.feature_extraction_utils import BatchFeature
from transformers.image_utils import ImageInput
from transformers.processing_utils import ProcessorMixin
from transformers.tokenization_utils_base import (
    PaddingStrategy,
    PreTokenizedInput,
    TextInput,
    TruncationStrategy,
)
from transformers.utils import TensorType
from trl import DPOConfig, DPOTrainer

2024-07-27 14:40:26.171910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-27 14:40:26.184461: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-27 14:40:26.188158: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-27 14:40:26.197149: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-27 14:40:26.999120: W tensorflow/compiler/tf2

[2024-07-27 14:40:28,669] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.1
 [WARNING]  using untested triton version (2.1.0), only 1.0.0 is known to be compatible


/tmp/own/conda/envs/Dev/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [2]:
class LlavaProcessor(ProcessorMixin):
    r"""
    Constructs a Llava processor which wraps a Llava image processor and a Llava tokenizer into a single processor.

    [`LlavaProcessor`] offers all the functionalities of [`CLIPImageProcessor`] and [`LlamaTokenizerFast`]. See the
    [`~LlavaProcessor.__call__`] and [`~LlavaProcessor.decode`] for more information.

    Args:
        image_processor ([`CLIPImageProcessor`], *optional*):
            The image processor is a required input.
        tokenizer ([`LlamaTokenizerFast`], *optional*):
            The tokenizer is a required input.
        chat_template (`str`, *optional*): A Jinja template which will be used to convert lists of messages
            in a chat into a tokenizable string.
    """

    attributes = ["image_processor", "tokenizer"]
    valid_kwargs = ["chat_template"]
    image_processor_class = "SiglipImageProcessor"
    tokenizer_class = "AutoTokenizer"

    def __init__(
        self, image_processor=None, tokenizer=None, chat_template=None, **kwargs
    ):
        super().__init__(image_processor, tokenizer, chat_template=chat_template)

    def __call__(
        self,
        text: Union[
            TextInput, PreTokenizedInput, List[TextInput], List[PreTokenizedInput]
        ] = None,
        images: ImageInput = None,
        padding: Union[bool, str, PaddingStrategy] = False,
        truncation: Union[bool, str, TruncationStrategy] = None,
        max_length=None,
        return_tensors: Optional[Union[str, TensorType]] = TensorType.PYTORCH,
    ) -> BatchFeature:
        """
        Main method to prepare for the model one or several sequences(s) and image(s). This method forwards the `text`
        and `kwargs` arguments to LlamaTokenizerFast's [`~LlamaTokenizerFast.__call__`] if `text` is not `None` to encode
        the text. To prepare the image(s), this method forwards the `images` and `kwrags` arguments to
        CLIPImageProcessor's [`~CLIPImageProcessor.__call__`] if `images` is not `None`. Please refer to the doctsring
        of the above two methods for more information.

        Args:
            text (`str`, `List[str]`, `List[List[str]]`):
                The sequence or batch of sequences to be encoded. Each sequence can be a string or a list of strings
                (pretokenized string). If the sequences are provided as list of strings (pretokenized), you must set
                `is_split_into_words=True` (to lift the ambiguity with a batch of sequences).
            images (`PIL.Image.Image`, `np.ndarray`, `torch.Tensor`, `List[PIL.Image.Image]`, `List[np.ndarray]`, `List[torch.Tensor]`):
                The image or batch of images to be prepared. Each image can be a PIL image, NumPy array or PyTorch
                tensor. Both channels-first and channels-last formats are supported.
            padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `False`):
                Select a strategy to pad the returned sequences (according to the model's padding side and padding
                index) among:
                - `True` or `'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
                  sequence if provided).
                - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
                  acceptable input length for the model if that argument is not provided.
                - `False` or `'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of different
                  lengths).
            max_length (`int`, *optional*):
                Maximum length of the returned list and optionally padding length (see above).
            truncation (`bool`, *optional*):
                Activates truncation to cut input sequences longer than `max_length` to `max_length`.
            return_tensors (`str` or [`~utils.TensorType`], *optional*):
                If set, will return tensors of a particular framework. Acceptable values are:

                - `'tf'`: Return TensorFlow `tf.constant` objects.
                - `'pt'`: Return PyTorch `torch.Tensor` objects.
                - `'np'`: Return NumPy `np.ndarray` objects.
                - `'jax'`: Return JAX `jnp.ndarray` objects.

        Returns:
            [`BatchFeature`]: A [`BatchFeature`] with the following fields:

            - **input_ids** -- List of token ids to be fed to a model. Returned when `text` is not `None`.
            - **attention_mask** -- List of indices specifying which tokens should be attended to by the model (when
              `return_attention_mask=True` or if *"attention_mask"* is in `self.model_input_names` and if `text` is not
              `None`).
            - **pixel_values** -- Pixel values to be fed to a model. Returned when `images` is not `None`.
        """
        if images is not None:
            # pixel_values = self.image_processor(images, return_tensors=return_tensors, size={"height": 768, "width": 768})["pixel_values"]
            images = images.convert("RGB")
            pixel_values = self.image_processor(images, return_tensors=return_tensors)[
                "pixel_values"
            ]
        else:
            pixel_values = None

        question = None
        answer = None

        user_name = "ユーザー: "
        assistant_name = "システム: "

        if user_name in text:
            question = text.split(user_name)[-1].strip()

            if assistant_name in text:
                answer = text.split(assistant_name)[-1].strip()
                question = question.replace(assistant_name + answer, "").strip()
        else:
            question = text

        inp = DEFAULT_IMAGE_TOKEN + "\n" + question
        conv = conv_templates["v1"].copy()
        conv.append_message(conv.roles[0], inp)
        conv.append_message(conv.roles[1], answer)
        prompt = conv.get_prompt()

        #print({"prompt": prompt})

        input_ids = tokenizer_image_token(
            prompt, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
        ).unsqueeze(0)

        input_ids = input_ids[:, :-1]

        attention_mask = input_ids.ne(self.tokenizer.pad_token_id).int()

        return BatchFeature(
            data={
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "pixel_values": pixel_values,
            }
        )

    # Copied from transformers.models.clip.processing_clip.CLIPProcessor.batch_decode with CLIP->Llama
    def batch_decode(self, *args, **kwargs):
        """
        This method forwards all its arguments to LlamaTokenizerFast's [`~PreTrainedTokenizer.batch_decode`]. Please
        refer to the docstring of this method for more information.
        """
        return self.tokenizer.batch_decode(*args, **kwargs)

    # Copied from transformers.models.clip.processing_clip.CLIPProcessor.decode with CLIP->Llama
    def decode(self, *args, **kwargs):
        """
        This method forwards all its arguments to LlamaTokenizerFast's [`~PreTrainedTokenizer.decode`]. Please refer to
        the docstring of this method for more information.
        """
        return self.tokenizer.decode(*args, **kwargs)

    @property
    # Copied from transformers.models.clip.processing_clip.CLIPProcessor.model_input_names
    def model_input_names(self):
        tokenizer_input_names = self.tokenizer.model_input_names
        image_processor_input_names = self.image_processor.model_input_names
        return list(dict.fromkeys(tokenizer_input_names + image_processor_input_names))

In [3]:
def main():
    # Load the model and processor
    """
    model = LlavaLlamaForCausalLM.from_pretrained(
        "team-hatakeyama-phase2/Tanuki-8B-vision-v4-checkpoint-26000",
        device_map="cuda",
        torch_dtype=torch.bfloat16,
        cache_dir="cache"
    )
    """

    model = LlavaGpt2ForCausalLM.from_pretrained(
        "hibikaze/finetune-llava-v1.5-japanese-gpt2-small_test-checkpoint-1200",
        device_map="cuda",
        torch_dtype=torch.bfloat16,
        cache_dir="cache",
        attn_implementation="eager"
    )

    ref_model = LlavaGpt2ForCausalLM.from_pretrained(
        "hibikaze/finetune-llava-v1.5-japanese-gpt2-small_test-checkpoint-1200",
        device_map="cuda",
        torch_dtype=torch.bfloat16,
        cache_dir="cache",
        attn_implementation="eager"
    )

    """
    processor = LlavaProcessor.from_pretrained(
        "team-hatakeyama-phase2/Tanuki-8B-vision-v4-checkpoint-26000", cache_dir="cache")
    """
    processor = LlavaProcessor.from_pretrained(
        "hibikaze/finetune-llava-v1.5-japanese-gpt2-small_test-checkpoint-1200", cache_dir="cache")

    #tokenizer = AutoTokenizer.from_pretrained("llava-hf/llava-1.5-7b-hf", cache_dir="cache")
    #LLAVA_CHAT_TEMPLATE = """{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{% if message['role'] == 'user' %}これは好奇心旺盛なユーザーと人工知能システムのチャットです。システムはユーザーの質問に親切、詳細、丁寧に答える。 ユーザー: {% else %}システム: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>\n{% endif %}{% endfor %}{% if message['role'] == 'user' %}{% else %}{{eos_token}}{% endif %}{% endfor %}{% if add_generation_prompt %}システム:  {% endif %}"""
    #processor.chat_template = LLAVA_CHAT_TEMPLATE
    #tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
    #processor.tokenizer = tokenizer

    print("===== processor =====")
    print(processor)
    #print("===== tokenizer =====")
    #print(tokenizer)
    print("=====================")
    
    # Load the dataset
    dataset = load_dataset("hibikaze/wit-dpo-test", split="train[:100%]", cache_dir="cache")

    def format(example):
        # Prepare the input for the chat template
        #prompt = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": example["question"]}]}]
        #chosen = [{"role": "assistant", "content": [{"type": "text", "text": example["chosen"]}]}]
        #rejected = [{"role": "assistant", "content": [{"type": "text", "text": example["rejected"]}]}]
        # Apply the chat template
        prompt = "ユーザー: " + example["question"]
        chosen = "システム: " + example["chosen"]
        rejected = "システム: " + example["rejected"]
        # Resize the image to ensure it fits within the maximum allowable
        # size of the processor to prevent OOM errors.
        #max_size = processor.image_processor.size["longest_edge"] // 2
        #example["image"].thumbnail((max_size, max_size))
        return {"images": example["image"], "prompt": prompt, "chosen": chosen, "rejected": rejected}
        #return {"images": [example["image"]], "prompt": prompt, "chosen": chosen, "rejected": rejected}
        #return {"images": [example["image"].convert('RGB')], "prompt": prompt, "chosen": chosen, "rejected": rejected}

    # Apply the formatting function to the dataset
    #dataset = dataset.map(format, remove_columns=dataset.column_names, num_proc=1)

    # .mapがうまく動かないことがあるため
    formatted_examples_list = [format(example) for example in tqdm(dataset)]

    print(formatted_examples_list[0])
    
    # リストを再びデータセット形式に変換
    formatted_dataset = Dataset.from_list(formatted_examples_list)

    dataset = formatted_dataset

    # Make sure that the images are decoded, it prevents from storing bytes.
    # More info here https://github.com/huggingface/blog/pull/2148#discussion_r1667400478
    #f = dataset.features
    #f["images"] = features.Sequence(features.Image(decode=True))
    #dataset = dataset.cast(f)

    # Train the model
    training_args = DPOConfig(
        output_dir="output_llava/dpo/llava-jp",
        #output_dir="idefics2-8b-dpo",
        bf16=True,
        gradient_checkpointing=True,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=32,
        num_train_epochs=1,
        dataset_num_proc=1,  # tokenization will use 32 processes
        dataloader_num_workers=1,  # data loading will use 32 workers
        logging_steps=2,
    )

    trainer = DPOTrainer(
        model,
        ref_model=ref_model,  # not needed when using peft
        args=training_args,
        train_dataset=dataset,
        tokenizer=processor,
        #peft_config=LoraConfig(target_modules="all-linear"),
    )

    trainer.train()

In [4]:
main()

You are using a model of type llava to instantiate a model of type llava-jp. This is not supported for all configurations of models and can yield errors.
/tmp/own/conda/envs/Dev/lib/python3.10/site-packages/torch/nn/modules/module.py:2025: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-meta parameter in the checkpoint to a meta '
/tmp/own/conda/envs/Dev/lib/python3.10/site-packages/torch/nn/modules/module.py:2025: UserWarning: for vision_model.embeddings.patch_embedding.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary t

===== processor =====
LlavaProcessor:
- image_processor: SiglipImageProcessor {
  "crop_size": {
    "height": 768,
    "width": 768
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "SiglipImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "processor_class": "LlavaProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 768,
    "width": 768
  }
}

- tokenizer: T5TokenizerFast(name_or_path='hibikaze/finetune-llava-v1.5-japanese-gpt2-small_test-checkpoint-1200', vocab_size=32000, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '[SEP]', 'pad_token': '<unk>', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder

100% 1808/1808 [00:01<00:00, 1396.12it/s]


{'images': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=300x225 at 0x7FA22851F970>, 'prompt': 'ユーザー: 画像に含まれている物が何であるか', 'chosen': 'システム: 多気城は、常陸国筑波郡多気にあった日本の城。多気山城・城山城とも称する。現存する史料や遺物が少なく、謎の城とされてきた。', 'rejected': 'システム: 多気城 (常陸国)'}


/home/docker/LLaVA-JP/external_libs/trl/trl/trainer/dpo_trainer.py:394: UserWarning: `max_length` is not set in the DPOConfig's init it will default to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/home/docker/LLaVA-JP/external_libs/trl/trl/trainer/dpo_trainer.py:407: UserWarning: `max_prompt_length` is not set in the DPOConfig's init it will default to `128` by default, but you should do it yourself in the future.
  warnings.warn(
/home/docker/LLaVA-JP/external_libs/trl/trl/trainer/dpo_trainer.py:442: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(
Parameter 'function'=<bound method DPOTrainer.tokenize_row of <trl.trainer.dpo_trainer.DPOTrainer object at 0x7fa22851ef50>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Ma

Map:   0%|          | 0/1808 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: hibikaze. Use `wandb login --relogin` to force relogin


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/tmp/own/conda/envs/Dev/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/tmp/own/conda/envs/Dev/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Could not estimate the numbe

Step,Training Loss
2,0.575600
4,0.321800
6,0.240000
8,0.194000
10,0.160500
12,0.162000
14,0.144300
16,0.140100
18,0.127900
20,0.112500
